# Formation of UXSSD Metadata

In [1]:
# importing all libraries
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import Audio, display



In [2]:
# reading the speaker information file and making a dataframe
uxssd_info_df=pd.read_csv("UXSSD_info.csv")
uxssd_info_df.head()

,SPEAKER-ID,GENDER,AGE-Y,AGE-M,AGE
0,01M,M,6,0,6.00
1,02M,M,10,1,10.08
2,03F,F,8,7,8.58
3,04M,M,8,11,8.92
4,05M,M,6,5,6.42


In [3]:
# checking the shape of the info dataframe
uxssd_info_df.shape

(8, 5)

In [4]:
# checking for any missing values
uxssd_info_df.isna().sum()

SPEAKER-ID    0
GENDER        0
AGE-Y         0
AGE-M         0
AGE           0
dtype: int64

In [5]:
# now forming a metatable of the data based inside core-uxssd directory
dataset = "uxssd" # storing dataset name as a string
dataset_root = Path("core-uxssd") # storing the folder path name

# sub-folders inside the root folder
core_dir = dataset_root/ "core"
lab_dir = dataset_root/ "speaker_labels"/ "lab"

rows = []

# now looping through dataset

# looping through each speaker folder
for speaker_dir in sorted(core_dir.iterdir()): # iterate is used to make a iterative generator for the for loop because core_dir is just a path
    if not speaker_dir.is_dir(): # check for folder only
        continue # skip if not found

    speaker_id = speaker_dir.name # taking the folder name

    for session_dir in sorted(speaker_dir.iterdir()):
        if not session_dir.is_dir(): # check for folder only
            continue

        session_id = session_dir.name

        for wav_path in sorted(session_dir.glob("*.wav")):
            file_id = wav_path.stem # skip the extension part

            txt_path = session_dir/ f"{file_id}.txt"
            full_id = f"{speaker_id}-{session_id}-{file_id}" # pattern to match the text file format
            lab_path = lab_dir/ f"{full_id}.lab"

            rows.append({
                "dataset_name": dataset,
                "speaker_id": speaker_id,
                "session_id": session_id,
                "file_id" : file_id, 
                "full_id": full_id,
                "raw_wav_path": str(wav_path),
                "txt_path": str(txt_path) if txt_path.exists() else None,
                "speaker_label_path": str(lab_path) if lab_path.exists() else None,
                "class_label": 1
            })

uxssd_metadata=pd.DataFrame(rows)
uxssd_metadata.head()


,dataset_name,speaker_id,session_id,file_id,full_id,raw_wav_path,txt_path,speaker_label_path,class_label
0,uxssd,01M,BL1,001A,01M-BL1-001A,core-uxssd/core/01M/BL1/001A.wav,core-uxssd/core/01M/BL1/001A.txt,core-uxssd/speaker_labels/lab/01M-BL1-001A.lab,1
1,uxssd,01M,BL1,002A,01M-BL1-002A,core-uxssd/core/01M/BL1/002A.wav,core-uxssd/core/01M/BL1/002A.txt,core-uxssd/speaker_labels/lab/01M-BL1-002A.lab,1
2,uxssd,01M,BL1,003A,01M-BL1-003A,core-uxssd/core/01M/BL1/003A.wav,core-uxssd/core/01M/BL1/003A.txt,core-uxssd/speaker_labels/lab/01M-BL1-003A.lab,1
3,uxssd,01M,BL1,004A,01M-BL1-004A,core-uxssd/core/01M/BL1/004A.wav,core-uxssd/core/01M/BL1/004A.txt,core-uxssd/speaker_labels/lab/01M-BL1-004A.lab,1
4,uxssd,01M,BL1,005A,01M-BL1-005A,core-uxssd/core/01M/BL1/005A.wav,core-uxssd/core/01M/BL1/005A.txt,core-uxssd/speaker_labels/lab/01M-BL1-005A.lab,1


In [6]:
# checking for missing values
uxssd_metadata.isna().sum()

dataset_name          0
speaker_id            0
session_id            0
file_id               0
full_id               0
raw_wav_path          0
txt_path              0
speaker_label_path    2
class_label           0
dtype: int64

In [7]:
# checking for column names
print(uxssd_info_df.columns)
print(uxssd_metadata.columns)

Index(['SPEAKER-ID', 'GENDER', 'AGE-Y', 'AGE-M', 'AGE'], dtype='str')
Index(['dataset_name', 'speaker_id', 'session_id', 'file_id', 'full_id',
       'raw_wav_path', 'txt_path', 'speaker_label_path', 'class_label'],
      dtype='str')


In [8]:
# renaming the speaker_id
uxssd_info_df=uxssd_info_df.rename(columns={"SPEAKER-ID":"speaker_id"})

In [9]:
print(uxssd_info_df.columns)
print(uxssd_metadata.columns)

Index(['speaker_id', 'GENDER', 'AGE-Y', 'AGE-M', 'AGE'], dtype='str')
Index(['dataset_name', 'speaker_id', 'session_id', 'file_id', 'full_id',
       'raw_wav_path', 'txt_path', 'speaker_label_path', 'class_label'],
      dtype='str')


In [10]:
# merging the info data with the metadata
uxssd_metadata=uxssd_metadata.merge(
    uxssd_info_df,
    on = "speaker_id",
    how = "left")

In [11]:
uxssd_metadata.head()

,dataset_name,speaker_id,session_id,file_id,full_id,raw_wav_path,txt_path,speaker_label_path,class_label,GENDER,AGE-Y,AGE-M,AGE
0,uxssd,01M,BL1,001A,01M-BL1-001A,core-uxssd/core/01M/BL1/001A.wav,core-uxssd/core/01M/BL1/001A.txt,core-uxssd/speaker_labels/lab/01M-BL1-001A.lab,1,M,6,0,6.0
1,uxssd,01M,BL1,002A,01M-BL1-002A,core-uxssd/core/01M/BL1/002A.wav,core-uxssd/core/01M/BL1/002A.txt,core-uxssd/speaker_labels/lab/01M-BL1-002A.lab,1,M,6,0,6.0
2,uxssd,01M,BL1,003A,01M-BL1-003A,core-uxssd/core/01M/BL1/003A.wav,core-uxssd/core/01M/BL1/003A.txt,core-uxssd/speaker_labels/lab/01M-BL1-003A.lab,1,M,6,0,6.0
3,uxssd,01M,BL1,004A,01M-BL1-004A,core-uxssd/core/01M/BL1/004A.wav,core-uxssd/core/01M/BL1/004A.txt,core-uxssd/speaker_labels/lab/01M-BL1-004A.lab,1,M,6,0,6.0
4,uxssd,01M,BL1,005A,01M-BL1-005A,core-uxssd/core/01M/BL1/005A.wav,core-uxssd/core/01M/BL1/005A.txt,core-uxssd/speaker_labels/lab/01M-BL1-005A.lab,1,M,6,0,6.0


In [12]:
uxssd_info_df.columns

Index(['speaker_id', 'GENDER', 'AGE-Y', 'AGE-M', 'AGE'], dtype='str')

In [13]:
uxssd_metadata.columns

Index(['dataset_name', 'speaker_id', 'session_id', 'file_id', 'full_id',
       'raw_wav_path', 'txt_path', 'speaker_label_path', 'class_label',
       'GENDER', 'AGE-Y', 'AGE-M', 'AGE'],
      dtype='str')

In [14]:
# rearranging the metadata
uxssd_metadata = uxssd_metadata[
    [
        "dataset_name",
        "speaker_id",
        "session_id",
        "file_id",
        "GENDER",
        "AGE-Y",
        "AGE-M",
        "AGE",
        "full_id",
        "raw_wav_path",
        "txt_path",
        "speaker_label_path",
        "class_label"
    ]
]
uxssd_metadata.head()

,dataset_name,speaker_id,session_id,file_id,GENDER,AGE-Y,AGE-M,AGE,full_id,raw_wav_path,txt_path,speaker_label_path,class_label
0,uxssd,01M,BL1,001A,M,6,0,6.0,01M-BL1-001A,core-uxssd/core/01M/BL1/001A.wav,core-uxssd/core/01M/BL1/001A.txt,core-uxssd/speaker_labels/lab/01M-BL1-001A.lab,1
1,uxssd,01M,BL1,002A,M,6,0,6.0,01M-BL1-002A,core-uxssd/core/01M/BL1/002A.wav,core-uxssd/core/01M/BL1/002A.txt,core-uxssd/speaker_labels/lab/01M-BL1-002A.lab,1
2,uxssd,01M,BL1,003A,M,6,0,6.0,01M-BL1-003A,core-uxssd/core/01M/BL1/003A.wav,core-uxssd/core/01M/BL1/003A.txt,core-uxssd/speaker_labels/lab/01M-BL1-003A.lab,1
3,uxssd,01M,BL1,004A,M,6,0,6.0,01M-BL1-004A,core-uxssd/core/01M/BL1/004A.wav,core-uxssd/core/01M/BL1/004A.txt,core-uxssd/speaker_labels/lab/01M-BL1-004A.lab,1
4,uxssd,01M,BL1,005A,M,6,0,6.0,01M-BL1-005A,core-uxssd/core/01M/BL1/005A.wav,core-uxssd/core/01M/BL1/005A.txt,core-uxssd/speaker_labels/lab/01M-BL1-005A.lab,1


In [15]:
# checking missing values again
uxssd_metadata.isna().sum()

dataset_name          0
speaker_id            0
session_id            0
file_id               0
GENDER                0
AGE-Y                 0
AGE-M                 0
AGE                   0
full_id               0
raw_wav_path          0
txt_path              0
speaker_label_path    2
class_label           0
dtype: int64

## Reading the prompt text for txt file

In [16]:
# reading text from the text files
def read_prompt_text(txt_path):
    if pd.isna(txt_path):
        return None
    try:
        with open(txt_path, "r", encoding="utf-8") as f:
            return f.readline().strip()
    except Exception:
        return None

uxssd_metadata["prompt_text"] = uxssd_metadata["txt_path"].apply(read_prompt_text) # function will apply on each value on the column
uxssd_metadata.head()

,dataset_name,speaker_id,session_id,file_id,GENDER,AGE-Y,AGE-M,AGE,full_id,raw_wav_path,txt_path,speaker_label_path,class_label,prompt_text
0,uxssd,01M,BL1,001A,M,6,0,6.0,01M-BL1-001A,core-uxssd/core/01M/BL1/001A.wav,core-uxssd/core/01M/BL1/001A.txt,core-uxssd/speaker_labels/lab/01M-BL1-001A.lab,1,elephant umbrella train swing
1,uxssd,01M,BL1,002A,M,6,0,6.0,01M-BL1-002A,core-uxssd/core/01M/BL1/002A.wav,core-uxssd/core/01M/BL1/002A.txt,core-uxssd/speaker_labels/lab/01M-BL1-002A.lab,1,bread duck giraffe five
2,uxssd,01M,BL1,003A,M,6,0,6.0,01M-BL1-003A,core-uxssd/core/01M/BL1/003A.wav,core-uxssd/core/01M/BL1/003A.txt,core-uxssd/speaker_labels/lab/01M-BL1-003A.lab,1,teeth watch orange school
3,uxssd,01M,BL1,004A,M,6,0,6.0,01M-BL1-004A,core-uxssd/core/01M/BL1/004A.wav,core-uxssd/core/01M/BL1/004A.txt,core-uxssd/speaker_labels/lab/01M-BL1-004A.lab,1,crab biscuits thank you helicopter
4,uxssd,01M,BL1,005A,M,6,0,6.0,01M-BL1-005A,core-uxssd/core/01M/BL1/005A.wav,core-uxssd/core/01M/BL1/005A.txt,core-uxssd/speaker_labels/lab/01M-BL1-005A.lab,1,egg splash square pig


## Audio Segmentation using speaker_label_path


In [17]:
# helper function for audio segmentation
def diarize_with_speaker_labels(wav_path, lab_path, output_path,
                                keep_label="CHILD",
                                margin_ms=20,
                                min_seg_ms=100):
    audio, sr = sf.read(wav_path)

    # convert stereo to mono if needed
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)

    margin_samples = round(sr * margin_ms / 1000)
    min_seg_samples = round(sr * min_seg_ms / 1000)

    kept_segments = []
    segment_info = []
    found_child = False
    found_slt = False

    with open(lab_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 3:
                continue

            start_htk = int(parts[0])
            end_htk = int(parts[1])
            label = parts[2].strip().upper()

            if label == "CHILD":
                found_child = True
            if label == "SLT":
                found_slt = True

            if label == keep_label.upper():
                start_sample = round(start_htk * sr / 10_000_000)
                end_sample = round(end_htk * sr / 10_000_000)

                # trim boundaries slightly
                start_sample += margin_samples
                end_sample -= margin_samples

                # clip bounds
                start_sample = max(0, start_sample)
                end_sample = min(len(audio), end_sample)

                if end_sample > start_sample:
                    seg = audio[start_sample:end_sample]
                    if len(seg) >= min_seg_samples:
                        kept_segments.append(seg)
                        segment_info.append((start_sample, end_sample))

    raw_duration_sec = len(audio) / sr

    if not kept_segments:
        return {
            "status": "no_child_segments",
            "sample_rate": sr,
            "raw_duration_sec": raw_duration_sec,
            "child_duration_sec": 0.0,
            "n_child_segments": 0,
            "has_child": found_child,
            "has_slt": found_slt
        }

    child_audio = np.concatenate(kept_segments)
    sf.write(output_path, child_audio, sr)

    return {
        "status": "processed",
        "sample_rate": sr,
        "raw_duration_sec": raw_duration_sec,
        "child_duration_sec": len(child_audio) / sr,
        "n_child_segments": len(kept_segments),
        "has_child": found_child,
        "has_slt": found_slt
    }

In [18]:
row = uxssd_metadata.iloc[0]

output_dir = Path("uxssd_processed_child_wav")
output_dir.mkdir(exist_ok=True)

output_path = output_dir / f"{row['full_id']}_child.wav"

result = diarize_with_speaker_labels(
    wav_path=row["raw_wav_path"],
    lab_path=row["speaker_label_path"],
    output_path=output_path
)

print(result)
print("Saved to:", output_path)

{'status': 'processed', 'sample_rate': 22050, 'raw_duration_sec': 11.470657596371883, 'child_duration_sec': 3.66, 'n_child_segments': 4, 'has_child': True, 'has_slt': True}
Saved to: uxssd_processed_child_wav/01M-BL1-001A_child.wav


In [19]:
uxssd_metadata.loc[0, "processed_child_wav_path"] = str(output_path)
uxssd_metadata.loc[0, "raw_duration_sec"] = result["raw_duration_sec"]
uxssd_metadata.loc[0, "child_duration_sec"] = result["child_duration_sec"]
uxssd_metadata.loc[0, "n_child_segments"] = result["n_child_segments"]
uxssd_metadata.loc[0, "has_child"] = result["has_child"]
uxssd_metadata.loc[0, "has_slt"] = result["has_slt"]
uxssd_metadata.loc[0, "status"] = result["status"]
uxssd_metadata.loc[0, "diarization_method"] = "provided_speaker_labels"

In [20]:
# testing first row instance
print("Before:")
display(Audio(row["raw_wav_path"]))

print("After:")
Audio(row["raw_wav_path"])
Audio(str(output_path))

Before:


After:


In [21]:
uxssd_metadata.head()

,dataset_name,speaker_id,session_id,file_id,GENDER,AGE-Y,AGE-M,AGE,full_id,raw_wav_path,...,class_label,prompt_text,processed_child_wav_path,raw_duration_sec,child_duration_sec,n_child_segments,has_child,has_slt,status,diarization_method
0,uxssd,01M,BL1,001A,M,6,0,6.0,01M-BL1-001A,core-uxssd/core/01M/BL1/001A.wav,...,1,elephant umbrella train swing,uxssd_processed_child_wav/01M-BL1-001A_child.wav,11.470658,3.66,4.0,True,True,processed,provided_speaker_labels
1,uxssd,01M,BL1,002A,M,6,0,6.0,01M-BL1-002A,core-uxssd/core/01M/BL1/002A.wav,...,1,bread duck giraffe five,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,uxssd,01M,BL1,003A,M,6,0,6.0,01M-BL1-003A,core-uxssd/core/01M/BL1/003A.wav,...,1,teeth watch orange school,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,uxssd,01M,BL1,004A,M,6,0,6.0,01M-BL1-004A,core-uxssd/core/01M/BL1/004A.wav,...,1,crab biscuits thank you helicopter,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,uxssd,01M,BL1,005A,M,6,0,6.0,01M-BL1-005A,core-uxssd/core/01M/BL1/005A.wav,...,1,egg splash square pig,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Now applying Segmentation on whole data frame

In [22]:
output_dir = Path("uxssd_processed_child_wav")
output_dir.mkdir(exist_ok=True)


for idx, row in uxssd_metadata.iterrows():
    wav_path = row["raw_wav_path"]
    lab_path = row["speaker_label_path"]

    if pd.isna(wav_path) or pd.isna(lab_path):
        uxssd_metadata.loc[idx, "status"] = "missing_path"
        continue

    output_path = output_dir / f"{row['full_id']}_child.wav"

    try:
        result = diarize_with_speaker_labels(
            wav_path=wav_path,
            lab_path=lab_path,
            output_path=output_path
        )

        uxssd_metadata.loc[idx, "processed_child_wav_path"] = str(output_path)
        uxssd_metadata.loc[idx, "raw_duration_sec"] = result["raw_duration_sec"]
        uxssd_metadata.loc[idx, "child_duration_sec"] = result["child_duration_sec"]
        uxssd_metadata.loc[idx, "n_child_segments"] = result["n_child_segments"]
        uxssd_metadata.loc[idx, "has_child"] = result["has_child"]
        uxssd_metadata.loc[idx, "has_slt"] = result["has_slt"]
        uxssd_metadata.loc[idx, "status"] = result["status"]
        uxssd_metadata.loc[idx, "diarization_method"] = "provided_speaker_labels"

    except Exception as e:
        uxssd_metadata.loc[idx, "status"] = f"error: {e}"

In [23]:
uxssd_metadata.head()

,dataset_name,speaker_id,session_id,file_id,GENDER,AGE-Y,AGE-M,AGE,full_id,raw_wav_path,...,class_label,prompt_text,processed_child_wav_path,raw_duration_sec,child_duration_sec,n_child_segments,has_child,has_slt,status,diarization_method
0,uxssd,01M,BL1,001A,M,6,0,6.0,01M-BL1-001A,core-uxssd/core/01M/BL1/001A.wav,...,1,elephant umbrella train swing,uxssd_processed_child_wav/01M-BL1-001A_child.wav,11.470658,3.660000,4.0,True,True,processed,provided_speaker_labels
1,uxssd,01M,BL1,002A,M,6,0,6.0,01M-BL1-002A,core-uxssd/core/01M/BL1/002A.wav,...,1,bread duck giraffe five,uxssd_processed_child_wav/01M-BL1-002A_child.wav,10.866939,2.359955,4.0,True,False,processed,provided_speaker_labels
2,uxssd,01M,BL1,003A,M,6,0,6.0,01M-BL1-003A,core-uxssd/core/01M/BL1/003A.wav,...,1,teeth watch orange school,uxssd_processed_child_wav/01M-BL1-003A_child.wav,10.263220,2.239909,5.0,True,False,processed,provided_speaker_labels
3,uxssd,01M,BL1,004A,M,6,0,6.0,01M-BL1-004A,core-uxssd/core/01M/BL1/004A.wav,...,1,crab biscuits thank you helicopter,uxssd_processed_child_wav/01M-BL1-004A_child.wav,13.885533,5.199955,5.0,True,True,processed,provided_speaker_labels
4,uxssd,01M,BL1,005A,M,6,0,6.0,01M-BL1-005A,core-uxssd/core/01M/BL1/005A.wav,...,1,egg splash square pig,uxssd_processed_child_wav/01M-BL1-005A_child.wav,9.659501,3.159909,4.0,True,False,processed,provided_speaker_labels


In [24]:
uxssd_metadata.columns

Index(['dataset_name', 'speaker_id', 'session_id', 'file_id', 'GENDER',
       'AGE-Y', 'AGE-M', 'AGE', 'full_id', 'raw_wav_path', 'txt_path',
       'speaker_label_path', 'class_label', 'prompt_text',
       'processed_child_wav_path', 'raw_duration_sec', 'child_duration_sec',
       'n_child_segments', 'has_child', 'has_slt', 'status',
       'diarization_method'],
      dtype='str')

In [25]:
row1=uxssd_metadata.iloc[1]
Audio(filename=row1["raw_wav_path"])


In [26]:
row1=uxssd_metadata.iloc[1]
Audio(filename=row1["processed_child_wav_path"])

In [27]:
row=uxssd_metadata.iloc[15]
Audio(filename=row["raw_wav_path"])

In [28]:
row=uxssd_metadata.iloc[15]
Audio(filename=row["processed_child_wav_path"])

In [29]:
# checking for null values
uxssd_metadata.isna().sum()

dataset_name                0
speaker_id                  0
session_id                  0
file_id                     0
GENDER                      0
AGE-Y                       0
AGE-M                       0
AGE                         0
full_id                     0
raw_wav_path                0
txt_path                    0
speaker_label_path          2
class_label                 0
prompt_text                 0
processed_child_wav_path    2
raw_duration_sec            2
child_duration_sec          2
n_child_segments            2
has_child                   2
has_slt                     2
status                      0
diarization_method          2
dtype: int64

In [30]:
uxssd_metadata.shape

(933, 22)

In [31]:
# checking for missing values instances
uxssd_metadata[uxssd_metadata["speaker_label_path"].isna()]

,dataset_name,speaker_id,session_id,file_id,GENDER,AGE-Y,AGE-M,AGE,full_id,raw_wav_path,...,class_label,prompt_text,processed_child_wav_path,raw_duration_sec,child_duration_sec,n_child_segments,has_child,has_slt,status,diarization_method
249,uxssd,03F,BL1,054B,F,8,7,8.58,03F-BL1-054B,core-uxssd/core/03F/BL1/054B.wav,...,1,p apa eepee opo,NaN,NaN,NaN,NaN,NaN,NaN,missing_path,NaN
483,uxssd,04M,BL1,126A,M,8,11,8.92,04M-BL1-126A,core-uxssd/core/04M/BL1/126A.wav,...,1,sock shoop shire shallow,NaN,NaN,NaN,NaN,NaN,NaN,missing_path,NaN


In [32]:
# removing missing values
uxssd_metadata.dropna(axis=0, inplace=True)
# checking the missing values
uxssd_metadata.isna().sum()


dataset_name                0
speaker_id                  0
session_id                  0
file_id                     0
GENDER                      0
AGE-Y                       0
AGE-M                       0
AGE                         0
full_id                     0
raw_wav_path                0
txt_path                    0
speaker_label_path          0
class_label                 0
prompt_text                 0
processed_child_wav_path    0
raw_duration_sec            0
child_duration_sec          0
n_child_segments            0
has_child                   0
has_slt                     0
status                      0
diarization_method          0
dtype: int64

In [33]:
# checking the label count with child part inside the speech
uxssd_metadata["has_child"].value_counts()

has_child
True     921
False     10
Name: count, dtype: int64

In [34]:
# we need to remove the false values to contain only the recodings which has child_labels
uxssd_metadata.drop(uxssd_metadata[uxssd_metadata["has_child"] == False].index, inplace=True)
uxssd_metadata["has_child"].value_counts()

has_child
True    921
Name: count, dtype: int64

In [35]:
# checking for prompt text
vc= uxssd_metadata["prompt_text"].value_counts(dropna=False)
print(vc.to_string())

prompt_text
DDK ppp ttt kkk ptk patticake                                                                                               23
crab biscuits thank you helicopter                                                                                          17
knife van ear this                                                                                                          17
book boy                                                                                                                    17
bread duck giraffe five                                                                                                     16
teeth watch orange school                                                                                                   16
egg splash square pig                                                                                                       16
gloves queen three frog                                                                            

### All the text prompts inside the UXSSD dataset were normal. So no need for any removal.

In [36]:
# saving the clean metadata file
uxssd_metadata.to_csv("uxssd_metadata_clean.csv", index=False)